# 第5章 NVIDIA Nemotron 3.5 LightningをColabで動かす

NVIDIA Nemotron 3.5 LightningのBF16版を4bit量子化で読み込み、通常チャットとGradio UIを実行します。Google Driveに十分な空き容量がない場合は、USE_DRIVE_CACHE=Falseにしてください。

> **実行方針**  
> Gradioは無条件にアップグレードせず、Colabに入っている版を使用します。


In [1]:
# =========================================
# コード5-2-1 実行環境の準備
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

# TorchとGradioはColab既定版を利用する。
%pip -q install -U transformers accelerate bitsandbytes

import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")

print("GPU:", torch.cuda.get_device_name(0))


GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-e9e73ddb-7e6d-4b22-341f-28aa5fd13a8d)
Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 195.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 81.9 MB/s eta 0:00:00
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
# =========================================
# コード5-2-2・5-2-3 Google Driveとキャッシュ設定
# =========================================
from google.colab import drive

drive.mount("/content/drive")

import os
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
USE_DRIVE_CACHE = True
DISABLE_XET = False
OFFLINE_MODE = False

if USE_DRIVE_CACHE:
    CACHE_DIR = PROJECT_DIR / "Program" / "hf_cache"
else:
    CACHE_DIR = Path("/content/hf_cache")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)

if DISABLE_XET:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

if OFFLINE_MODE:
    os.environ["HF_HUB_OFFLINE"] = "1"

usage = shutil.disk_usage(CACHE_DIR)
print("CACHE_DIR:", CACHE_DIR)
print(f"free space: {usage.free / 1024**3:.1f} GB")

if USE_DRIVE_CACHE and usage.free < 35 * 1024**3:
    print("WARNING: Nemotron 3.5 Lightningのキャッシュには十分な空き容量が必要です。")


Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Program/hf_cache
free space: 179.1 GB


- 非常に時間がかかります

In [3]:
# =========================================
# コード5-2-4 Nemotron 3.5 Lightningモデルと応答生成関数
# =========================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    cache_dir=str(CACHE_DIR),
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)
model.eval()

INPUT_DEVICE = model.get_input_embeddings().weight.device

@torch.inference_mode()
def chat_generate(
    messages,
    max_new_tokens=256,
    do_sample=False,
    temperature=0.7,
    top_p=0.9,
):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,   # ← ここを追加
    ).to(INPUT_DEVICE)

    generation_kwargs = {
        **inputs,
        "max_new_tokens": int(max_new_tokens),
        "do_sample": bool(do_sample),
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "repetition_penalty": 1.05,
        "use_cache": True,
    }

    if do_sample:
        generation_kwargs["temperature"] = float(temperature)
        generation_kwargs["top_p"] = float(top_p)

    outputs = model.generate(**generation_kwargs)
    generated_ids = outputs[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

print("model loaded:", MODEL_ID)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/401 [00:00<?, ?it/s]

model loaded: nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16


In [4]:
# =========================================
# コード5-2-4 動作確認
# =========================================
messages = [
    {
        "role": "system",
        "content": "あなたは日本語で簡潔に答える親切なアシスタントです。",
    },
    {
        "role": "user",
        "content": "日本語で1文だけ自己紹介してください。",
    },
]

print(chat_generate(messages, max_new_tokens=96, do_sample=False))


私の名前はNemotronで、NVIDIAの研究者たちによって訓練された言語モデルです。


In [5]:
# =========================================
# コード5-2-5 Gradioを用いたローカルLLMチャットUI
# =========================================
import gradio as gr
import inspect

def make_messages_chatbot(**kwargs):
    if "type" in inspect.signature(gr.Chatbot).parameters:
        kwargs["type"] = "messages"
    return gr.Chatbot(**kwargs)

print("gradio:", gr.__version__)

def gr_chat(history, user_msg):
    history = history or []
    user_msg = (user_msg or "").strip()

    if not user_msg:
        return history, "", history

    messages = history[-8:] + [
        {
            "role": "user",
            "content": user_msg,
        }
    ]

    reply = chat_generate(
        messages,
        max_new_tokens=256,
        do_sample=False,
    )

    new_history = history + [
        {
            "role": "user",
            "content": user_msg,
        },
        {
            "role": "assistant",
            "content": reply,
        },
    ]

    return new_history, "", new_history


with gr.Blocks(title="Local LLM Chat") as chat_demo:
    gr.Markdown(
      "## Local LLM Chat\n"
    )
    chatbot = make_messages_chatbot(
        label="Chat",
        show_label=False,
        sanitize_html=True,
    )

    chat_state = gr.State([])

    user_box = gr.Textbox(
        placeholder="質問を入力してください。",
        label="",
    )

    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear")

    send_btn.click(
        gr_chat,
        inputs=[chat_state, user_box],
        outputs=[chat_state, user_box, chatbot],
        queue=False,
    )

    clear_btn.click(
        lambda: ([], "", []),
        outputs=[chat_state, user_box, chatbot],
    )

print("WARNING: share=Trueで公開URLが作成されます。")
chat_demo.launch(
    share=True,
    inline=True,
    debug=False,
)

gradio: 6.20.0
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0b14549e7ae3c1481e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
